# 18. 4Sum

[Problem](https://leetcode.com/problems/4sum/) · difficulty: medium

3Sum with one more loop around it. Two approaches that differ in a single line — what collects
the answer — and the question this notebook settles is whether that line matters at all.


In [ ]:
import pathlib, sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'lc').is_dir())
PROBLEM = ROOT / 'problems' / '0018-4sum'
sys.path.insert(0, str(ROOT))

from lc.harness import load_solutions

solutions = load_solutions(PROBLEM)
[s.__name__ for s in solutions]


## Summary

| Approach | Time | Space | Collects into | LeetCode |
|---|---|---|---|---|
| `SolutionBisectJumpSet` | O(n³) | O(t) | a set | 8 ms |
| `SolutionBisectJumpList` | O(n³) | O(1) | a list | 20 ms |

`t` is the size of the answer.


## Does the set remove anything?

The set is only worth its memory if the scan can reach the same quadruplet twice. It cannot:
all four positions step over equal values — `i` and `j` skip repeats of the previous pin, and
`left`/`right` skip repeats after a hit — so each quadruplet is produced exactly once.

Rather than argue it, count what the list version emits before any deduplication.


In [ ]:
from collections import Counter
import random

by_name = {s.__name__: s for s in solutions}

random.seed(18)
worst = None
for _ in range(2000):
    nums = [random.randint(-8, 8) for _ in range(random.randint(4, 14))]
    target = random.randint(-12, 12)
    emitted = by_name['SolutionBisectJumpList']().fourSum(list(nums), target)
    counts = Counter(tuple(q) for q in emitted)
    repeats = {q: c for q, c in counts.items() if c > 1}
    if repeats:
        worst = (nums, target, repeats)
        break

print('a quadruplet emitted twice:', worst or 'never, in 2000 random arrays')


So the two approaches return the same answer on every input, and the set is doing no work.
Confirming that on the largest input the constraints allow:


In [ ]:
random.seed(18)
DENSE = [random.randint(-20, 20) for _ in range(200)]

answers = {s.__name__: s().fourSum(list(DENSE), 0) for s in solutions}
for name, answer in answers.items():
    print(f'{name:<26} {len(answer):>5} quadruplets')

def canonical(quadruplets):
    return sorted(sorted(q) for q in quadruplets)

print('identical answers:', canonical(answers['SolutionBisectJumpSet']) == canonical(answers['SolutionBisectJumpList']))


## What the two actually cost

Four shapes at n = 200, the constraint's limit. The first has values spread across ±10⁹, so the
answer is empty and the scan runs to the end; the others concentrate the values.


In [ ]:
import time

random.seed(18)
SHAPES = {
    'wide range, no answer': ([random.randint(-10**9, 10**9) for _ in range(200)], 0),
    'values in ±20': (DENSE, 0),
    'all zeros': ([0] * 200, 0),
    'ascending, answer first': (list(range(1, 201)), 10),
}

def timed(solution, nums, target, runs=7):
    samples = []
    for _ in range(runs):
        start = time.perf_counter()
        solution().fourSum(list(nums), target)
        samples.append((time.perf_counter() - start) * 1e3)
    return min(samples)

print(f"{'approach':<28}" + ''.join(f'{name:>26}' for name in SHAPES))
for solution in solutions:
    row = ''.join(f'{timed(solution, nums, target):23.2f} ms' for nums, target in SHAPES.values())
    print(f'{solution.__name__:<28}{row}')


Within noise of each other. The set costs a hash per quadruplet found and buys nothing, so the
list is the better default — but the difference is far too small to explain the judge's 8 ms
against 20 ms. Those were two submissions at different moments, and LeetCode's timer varies by
more than this between runs of identical code.

Which is the real lesson here: **the two classes in this file were byte-identical** until this
notebook was written. The earlier repo benchmark that put them 1.04x apart was timing one
program against a copy of itself. A benchmark cannot tell you that — a diff can, in a second.


## Where the time really goes

Not in the collection, but in the pruning. Each pinned pair is skipped outright when its best
possible partners cannot reach the target, and the loop breaks when even the smallest partners
overshoot. Counting how many of the O(n²) pinned pairs survive shows why a cubic scan finishes
at all.


In [ ]:
def surviving_pairs(nums, target):
    nums = sorted(nums)
    n = len(nums)
    considered = survived = 0
    for i in range(n - 3):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        if nums[i] + nums[i+1] + nums[i+2] + nums[i+3] > target:
            break
        if nums[i] + nums[n-1] + nums[n-2] + nums[n-3] < target:
            continue
        for j in range(i + 1, n - 2):
            considered += 1
            if j > i + 1 and nums[j] == nums[j - 1]:
                continue
            pinned = nums[i] + nums[j]
            if pinned + nums[j+1] + nums[j+2] > target:
                break
            if pinned + nums[n-1] + nums[n-2] < target:
                continue
            survived += 1
    return considered, survived

for label, (nums, target) in SHAPES.items():
    considered, survived = surviving_pairs(nums, target)
    share = survived / considered * 100 if considered else 0
    print(f'{label:<26} {survived:>6} of {considered:>6} pinned pairs survive  ({share:4.1f}%)')


## Takeaway

- Diff before you benchmark. Two identical implementations will happily produce a 1.04x
  ranking, and nothing in the measurement will tell you they are the same file.
- A set used for deduplication is worth keeping only if you can produce an input where it
  removes something. Here the skips on all four positions already guarantee uniqueness.
- The judge's millisecond figure is a sample, not a measurement. 8 ms against 20 ms for programs
  that measure within 5% of each other locally is the timer talking, not the code.
